# Dyson Protocol Language Module Guide

This notebook provides a comprehensive guide to the Dyson Protocol Language Module (dyslang), which offers a secure and sandboxed Python execution environment on the blockchain. Through practical examples, we'll explore how to interact with the blockchain state, manage gas consumption, and leverage the powerful features of the `dys` module to build robust decentralized applications.

## Introduction to dyslang

The dyslang module serves as the backbone for on-chain Python execution in the Dyson Protocol. It provides a set of functions that enable scripts to:

- **Query Chain State**: Access account balances, contract data, and module parameters
- **Execute Transactions**: Send tokens, create contracts, and interact with other modules
- **Manage Resources**: Monitor gas consumption and execution limits
- **Access Context**: Retrieve information about the current script, executor, and block
- **Emit Events**: Produce blockchain events that can be indexed and monitored
- **Evaluate Code**: Execute dynamic Python code within a controlled environment

Let's dive into these features with practical examples.

## Setting Up

Before we start, let's set up our environment by defining our test accounts:

In [ ]:
# Get addresses of our test accounts
out = ! dysond keys show -a alice
print(out)  
ALICE_ADDRESS = out[0]
[BOB_ADDRESS] = ! dysond keys show -a bob

print(f"Using alice address: {ALICE_ADDRESS}")
print(f"Using bob address: {BOB_ADDRESS}")

## Chain Interaction

The dyslang module provides two primary functions for interacting with the blockchain: `_query` and `_msg`.

- `_query`: Used to query the blockchain state (read-only operations)
- `_msg`: Used to submit transactions that modify the blockchain state

Let's explore these functions with practical examples.

### Querying Account Balances

One common operation is to query an account's balance. Let's create a script that queries the balance of an account:

In [ ]:
import json
import tempfile
import os

# Create a script that queries the account balance
query_script = '''
from dys import _query, get_script_address
import json

def query_balance():
    # Query the script's own balance
    script_address = get_script_address()
    response = _query({
        "@type": "/cosmos.bank.v1beta1.QueryBalanceRequest",
        "address": script_address,
        "denom": "udys"
    })
    return response
'''

# Write the script to a temporary file and ensure it is deleted after use
with tempfile.NamedTemporaryFile("w+", suffix=".py", delete=True) as f:
    f.write(query_script)
    f.flush()
    # Execute the script using dysond query script run
    out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name query_balance --extra-code-path {f.name} -o json
out = '\n'.join(out)
print(out)
result = json.loads(out)
json_result = json.loads(result['result'])['result']

assert 'balance' in json_result, "Balance not found in the result"
print(json.dumps(json_result['balance'], indent=2))

### Querying Multiple Account Balances

Let's create a more advanced script that queries the balances of multiple accounts:

In [ ]:
# Create a script that queries multiple account balances
import tempfile
import json

query_multi_script = f'''
from dys import _query
import json

def query_multiple_balances():
    # Define the addresses to query
    alice_address = "{ALICE_ADDRESS}"
    bob_address = "{BOB_ADDRESS}"
    
    # Query Alice's balance
    alice_balance = _query({{
        "@type": "/cosmos.bank.v1beta1.QueryBalanceRequest",
        "address": alice_address,
        "denom": "udys"
    }})
    
    # Query Bob's balance
    bob_balance = _query({{
        "@type": "/cosmos.bank.v1beta1.QueryBalanceRequest",
        "address": bob_address,
        "denom": "udys"
    }})
    
    # Return both balances
    return {{
        "alice_balance": alice_balance,
        "bob_balance": bob_balance
    }}
'''

# Write the script to a temporary file and ensure it is deleted after use
with tempfile.NamedTemporaryFile("w+", suffix=".py", delete=True) as f:
    f.write(query_multi_script)
    f.flush()
    path = f.name
    # Execute the script using dysond query script run
    out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name query_multiple_balances --extra-code-path {path} -o json
out = '\n'.join(out)
print(out)
result = json.loads(out)
json_result = json.loads(result['result'])['result']
assert 'alice_balance' in json_result, "Alice's balance not found in the result"
assert 'bob_balance' in json_result, "Bob's balance not found in the result"
print(json.dumps(json_result, indent=2))

## Gas Management

In blockchain environments, computational resources are metered using a concept called "gas". The dyslang module provides several functions to help you monitor and manage gas consumption in your scripts.

### Monitoring Gas Consumption

Let's create a script that measures the gas consumed by various operations:

In [ ]:
# Create a script to benchmark gas consumption
gas_benchmark_script = '''
from dys import _query, get_gas_consumed, get_script_address, get_gas_limit
import json

def benchmark_gas(iterations=5):
    # Start tracking gas
    initial_gas = get_gas_consumed()
    
    # Perform a query that consumes gas
    script_address = get_script_address()
    balance_response = _query({
        "@type": "/cosmos.bank.v1beta1.QueryBalanceRequest",
        "address": script_address,
        "denom": "udys"
    })
    
    # Check gas after query
    after_query_gas = get_gas_consumed()
    
    # Run some iterations to measure their gas cost
    for i in range(iterations):
        print(f"Iteration {i+1} of {iterations}")
    
    # Check final gas consumption
    final_gas = get_gas_consumed()
    
    # Calculate gas used by different operations
    query_gas = after_query_gas - initial_gas
    iterations_gas = final_gas - after_query_gas
    
    return {
        "initial_gas": initial_gas,
        "after_query_gas": after_query_gas,
        "final_gas": final_gas,
        "query_gas": query_gas,
        "iterations_gas": iterations_gas,
        "per_iteration": iterations_gas / iterations
    }
'''

# Save and execute the script
with open('/tmp/gas_benchmark.py', 'w') as f:
    f.write(gas_benchmark_script)

out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name benchmark_gas --extra-code-path /tmp/gas_benchmark.py -o json
out = '\n'.join(out)
print(out)
result = json.loads(out)

# Extract and display the gas measurements
gas_metrics = json.loads(result['result'])['result']
assert 'initial_gas' in gas_metrics, "Initial gas not found in the result: " + str(gas_metrics)
print(f"Gas report for benchmark operations:")
print(f"- Initial gas consumed: {gas_metrics['initial_gas']}")
print(f"- Gas after query: {gas_metrics['after_query_gas']}")
print(f"- Gas after iterations: {gas_metrics['final_gas']}")
print(f"- Total gas for query: {gas_metrics['query_gas']}")
print(f"- Total gas for iterations: {gas_metrics['iterations_gas']}")
print(f"- Average gas per iteration: {gas_metrics['per_iteration']}")

### Gas Limits

Each execution has a gas limit to prevent infinite loops or excessive computation. Let's check the gas limit for our execution:

In [ ]:
# Create a script to check the gas limit
gas_limit_script = '''
from dys import get_gas_limit

def check_limit():
    # Check the gas limit for the current execution
    limit = get_gas_limit()
    return {"gas_limit": limit}
'''

# Save and execute the script
with open('/tmp/gas_limit.py', 'w') as f:
    f.write(gas_limit_script)

out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name check_limit --extra-code-path /tmp/gas_limit.py -o json
out = '\n'.join(out)
result = json.loads(out)

# Extract and display the gas limit 
gas_limit = json.loads(result['result'])['result']['gas_limit']
print(f"Gas limit for this execution: {gas_limit}")

### Node Execution Tracking

The dyslang module tracks the execution of Python AST nodes. This is useful for understanding the computational complexity of your scripts:

In [ ]:
# Create a script to measure node execution
node_count_script = '''
from dys import _query, get_nodes_called, get_script_address
import json

def count_nodes():
    """
    Demonstrate node counting by performing operations of varying complexity
    """
    # Simple operations
    a = 1 + 2
    
    # More complex operation that will use more nodes
    script_address = get_script_address()
    _query({
        "@type": "/cosmos.bank.v1beta1.QueryBalanceRequest",
        "address": script_address,
        "denom": "udys"
    })
    
    # Complex calculation with loop
    result = 0
    for i in range(10):
        result += i * 2
    
    # Get the count of AST nodes evaluated
    nodes_called = get_nodes_called()
    
    return {
        "nodes_called": nodes_called,
        "calculation_result": result
    }
'''

# Save and execute the script
with open('/tmp/node_count.py', 'w') as f:
    f.write(node_count_script)

out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name count_nodes --extra-code-path /tmp/node_count.py -o json
out = '\n'.join(out)
result = json.loads(out)


# Extract and display the node metrics
node_metrics = json.loads(result['result'])['result']
assert 'nodes_called' in node_metrics, "Nodes called not found in the result: " + str(node_metrics)

print(f"Node execution metrics:")
print(f"- Nodes called: {node_metrics['nodes_called']}")
print(f"- Calculation result: {node_metrics['calculation_result']}")

### Memory Usage Tracking

The dyslang module also tracks memory usage through the `get_cumulative_size()` function:

In [ ]:
# Create a script to check memory usage
memory_script = '''
from dys import get_cumulative_size

def check_memory():
    # Check the cumulative memory size used
    size = get_cumulative_size()
    return {"memory_used": size}
'''

# Save and execute the script
with open('/tmp/memory_check.py', 'w') as f:
    f.write(memory_script)

out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name check_memory --extra-code-path /tmp/memory_check.py -o json
out = '\n'.join(out)
result = json.loads(out)

# Extract and display the memory usage
memory_used = json.loads(result['result'])['result']
assert 'memory_used' in memory_used, "Memory used not found in the result: " + str(memory_used)
print(f"Memory usage: {memory_used['memory_used']} bytes")

## Context Information

The dyslang module provides several functions to access contextual information about the current execution environment, including the script's address, the executor's address, and block information.

### Script and Executor Addresses

Let's create a script that retrieves information about the script's own address and the address of the account executing the script:

In [ ]:
# Create a script to get address information
address_script = '''
from dys import get_executor_address, get_script_address

def who_called_me():
    """Returns information about who executed this script"""
    # Get the script's own address
    script_address = get_script_address()
    
    # Get the address of who called this script
    caller_address = get_executor_address()
    
    # Check if the script was called by its owner
    is_self_call = script_address == caller_address
    
    return {
        "script_address": script_address,
        "caller_address": caller_address,
        "is_self_call": is_self_call
    }
'''

# Save and execute the script
with open('/tmp/address_info.py', 'w') as f:
    f.write(address_script)

# Execute with Bob calling Alice's script
out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name who_called_me --extra-code-path /tmp/address_info.py -o json
out = '\n'.join(out)
print(out)
result = json.loads(out)


# Extract and display the address information
address_info = json.loads(result['result'])['result']
assert 'script_address' in address_info, "Script address not found in the result: " + str(address_info)
assert 'caller_address' in address_info, "Caller address not found in the result: " + str(address_info)
assert 'is_self_call' in address_info, "Is self call not found in the result: " + str(address_info)
print(f"Script Address: {address_info['script_address']}")
print(f"Executor Address: {address_info['caller_address']}")
print(f"Self-execution: {address_info['is_self_call']}")

### Block Information

Let's retrieve information about the current block:

In [ ]:
# Create a script to get block information
block_info_script = '''
from dys import get_block_info

def show_block_info():
    """Get basic block information"""
    block = get_block_info()
    return {
        "height": block.get("height"),
        "chain_id": block.get("chain_id"),
        "time": block.get("time")
    }
'''

# Save and execute the script
with open('/tmp/block_info.py', 'w') as f:
    f.write(block_info_script)

out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name show_block_info --extra-code-path /tmp/block_info.py -o json
out = '\n'.join(out)
result = json.loads(out)

# Extract and display the block information
block_info = json.loads(result['result'])['result']
assert 'height' in block_info and block_info['height'] is not None, "Height not found in the result: " + str(block_info)
assert 'chain_id' in block_info and block_info['chain_id'] is not None, "Chain ID not found in the result: " + str(block_info)
assert 'time' in block_info and block_info['time'] is not None, "Time not found in the result: " + str(block_info)
print(f"Block Information:")
print(f"- Height: {block_info['height']}")
print(f"- Chain ID: {block_info['chain_id']}")
print(f"- Time: {block_info['time']}")

## Transaction Data

The dyslang module allows scripts to access information about attached messages in transactions. This is particularly useful for scripts that need to process multiple operations in a single transaction.

### Attached Messages

Let's create a script that checks for attached messages:

In [ ]:
# Create a script to check for attached messages
import shlex
import json

msg1 = shlex.quote(json.dumps({
        "@type":"/cosmos.bank.v1beta1.MsgSend",
        "from_address": ALICE_ADDRESS,
        "to_address": BOB_ADDRESS,
        "amount":[{"denom":"udys","amount":"12"}]
    }))


msg2 = shlex.quote(json.dumps({
    "@type":"/cosmos.bank.v1beta1.MsgSend",
    "from_address": ALICE_ADDRESS   ,
    "to_address": BOB_ADDRESS,
    "amount":[{"denom":"udys","amount":"34"}]
}))

attached_msgs_script = '''
from dys import get_attached_messages, get_attached_msg_results

def check_messages():
    # Access attached messages
    attached_messages = get_attached_messages()
    attached_msg_results = get_attached_msg_results()
    return {"attached_messages": attached_messages, "attached_msg_results": attached_msg_results}
'''

# Save and execute the script
with open('/tmp/attached_msgs.py', 'w') as f:
    f.write(attached_msgs_script)

out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name check_messages --extra-code-path /tmp/attached_msgs.py -o json --attached-message {msg1} --attached-message {msg2} 

out = '\n'.join(out)
print(out)
result = json.loads(out)


# Extract and display the attached messages

results = json.loads(result['result'])['result']
print(results)
assert 'attached_messages' in results, "Attached messages not found in the result: " + str(results)
assert 'attached_msg_results' in results, "Attached msg results not found in the result: " + str(results)
for m, r in zip(results['attached_messages'], results['attached_msg_results']):
    print(f"Message: {m}")
    print(f"Result: {r}")

## Events and Evaluation

The dyslang module provides functions to emit blockchain events and evaluate dynamic code at runtime.

### Emitting Events

Let's create a script that emits a custom event to the blockchain:

In [ ]:
# Create a script to emit an event
emit_event_script = '''
from dys import emit_event

def emit_test_event():
    # Emit a custom event, none is a success or an exception is raised
    emit_event("payment_processed", "success")
    emit_event("foo", '123123')
    return {"event_emitted": True}
'''

# Save and execute the script
with open('/tmp/emit_event.py', 'w') as f:
    f.write(emit_event_script)


out = ! dysond tx script exec --script-address {ALICE_ADDRESS} --from {ALICE_ADDRESS} --function-name emit_test_event --extra-code-path /tmp/emit_event.py -y --gas "10000000" | dysond q wait-tx -o json
out = '\n'.join(out)
try:
    result = json.loads(out)
except json.JSONDecodeError:
    print("Error decoding JSON:", out)
    raise
# Quering script run does not emit events
print(json.dumps(result, indent=2))
# make the events more readable
events = {}
for e in result['events']:
    event_type = e['type']
    events.setdefault(event_type, {})
    for a in e['attributes']:
        events[event_type][a['key']] = a['value']
            
print(json.dumps(events, indent=2))
assert events['dysonprotocol.script.v1.EventScriptEvent']['key'] == '"foo"', "Event foo not found in the result: " + str(events)
assert events['dysonprotocol.script.v1.EventScriptEvent']['value'] == '"123123"', "Event value not found in the result: " + str(events)

### Dynamic Code Evaluation

The `dys_eval` function allows you to evaluate Python code dynamically at runtime. This is a powerful feature that enables creating flexible and adaptable scripts:

In [ ]:
# Create a script for dynamic code evaluation demonstrating ALL dys_eval parameters
dys_eval_script = '''
from dys import dys_eval

def demonstrate_dys_eval():
    """
    Demonstrate all dys_eval parameters:
    - code: the code string to evaluate
    - scope: dict of variables available during evaluation
    - max_node_calls: limit AST node evaluations (prevents runaway loops)
    - max_scope_size: limit scope size (prevents memory exhaustion)
    - track_func: callback after each node evaluation
    - module_dict: custom modules to make available for import
    """
    results = {}
    
    # 1. BASIC: code parameter only
    results["basic_arithmetic"] = dys_eval("2 + 3 * 4")
    results["basic_string"] = dys_eval("'hello ' + 'world'.upper()")
    
    # 2. SCOPE: pass variables into the evaluation context
    local_scope = {'x': 10, 'y': 5, 'data': [1, 2, 3]}
    results["scope_multiply"] = dys_eval("x * y", scope=local_scope)
    results["scope_list_sum"] = dys_eval("sum(data)", scope=local_scope)
    results["scope_complex"] = dys_eval("x + y + len(data)", scope=local_scope)
    
    # 3. MAX_NODE_CALLS: limit computation to prevent infinite loops
    # Small limit that allows simple operations
    results["node_limit_ok"] = dys_eval("1 + 2 + 3", max_node_calls=100)
    
    # Attempting a loop that exceeds node limit will raise Exception
    try:
        dys_eval("sum([i for i in range(1000)])", max_node_calls=50)
        results["node_limit_exceeded"] = "should have failed"
    except Exception as e:
        results["node_limit_exceeded"] = f"Caught: {str(e)[:30]}"
    
    # 4. MAX_SCOPE_SIZE: limit memory usage by restricting scope size
    results["scope_size_ok"] = dys_eval("x = 1; y = 2; x + y", max_scope_size=1000)
    
    # Creating many variables exceeds scope size limit
    try:
        dys_eval("[i for i in range(10000)]", max_scope_size=100)
        results["scope_size_exceeded"] = "should have failed"
    except Exception as e:
        results["scope_size_exceeded"] = f"Caught: {str(e)[:40]}"
    
    # 5. TRACK_FUNC: callback for each AST node evaluation
    node_count = {"count": 0, "types": set()}
    def my_tracker(lineno, col_offset, end_lineno, end_col_offset, node_type):
        node_count["count"] += 1
        node_count["types"].add(node_type)
    
    dys_eval("a = 1; b = 2; a + b", track_func=my_tracker)
    results["track_func_count"] = node_count["count"]
    results["track_func_types"] = list(node_count["types"])[:10]  # First 10 node types
    
    # 6. MODULE_DICT: inject custom modules into the sandbox
    # Define a custom "math_utils" module with helper functions
    custom_modules = {
        "math_utils": {
            "double": lambda x: x * 2,
            "triple": lambda x: x * 3,
            "add_ten": lambda x: x + 10,
        },
        "string_utils": {
            "shout": lambda s: s.upper() + "!",
            "whisper": lambda s: s.lower() + "...",
        }
    }
    
    results["module_math"] = dys_eval(
        "from math_utils import double, triple; double(5) + triple(3)",
        module_dict=custom_modules
    )
    results["module_string"] = dys_eval(
        "from string_utils import shout; shout('hello')",
        module_dict=custom_modules
    )
    
    # 7. COMBINED: use multiple parameters together
    combined_scope = {"base": 100}
    combined_modules = {"ops": {"halve": lambda x: x // 2}}
    combined_count = {"n": 0}
    def combined_tracker(lineno, col_offset, end_lineno, end_col_offset, node_type):
        combined_count["n"] += 1
    
    results["combined"] = dys_eval(
        "from ops import halve; halve(base)",
        scope=combined_scope,
        max_node_calls=500,
        max_scope_size=500,
        track_func=combined_tracker,
        module_dict=combined_modules
    )
    results["combined_nodes_used"] = combined_count["n"]
    
    return results
'''

# Save and execute the script
with open('/tmp/dys_eval.py', 'w') as f:
    f.write(dys_eval_script)

out = ! dysond q script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name demonstrate_dys_eval --extra-code-path /tmp/dys_eval.py -o json
out = '\n'.join(out)
try:
    json_out = json.loads(out)
except json.JSONDecodeError:
    print("Error decoding JSON:", out)
    raise

# Extract and display the evaluation results
eval_results = json.loads(json_out['result'])['result']

print("=== dys_eval Parameter Demonstration ===\n")

print("1. BASIC (code only):")
print(f"   arithmetic: 2 + 3 * 4 = {eval_results['basic_arithmetic']}")
print(f"   string: 'hello ' + 'world'.upper() = {eval_results['basic_string']}")

print("\n2. SCOPE (variables in evaluation context):")
print(f"   x * y (x=10, y=5) = {eval_results['scope_multiply']}")
print(f"   sum(data) (data=[1,2,3]) = {eval_results['scope_list_sum']}")
print(f"   x + y + len(data) = {eval_results['scope_complex']}")

print("\n3. MAX_NODE_CALLS (limit computation):")
print(f"   1 + 2 + 3 (limit=100) = {eval_results['node_limit_ok']}")
print(f"   large loop (limit=50) = {eval_results['node_limit_exceeded']}")

print("\n4. MAX_SCOPE_SIZE (limit memory):")
print(f"   small scope (limit=1000) = {eval_results['scope_size_ok']}")
print(f"   large list (limit=100) = {eval_results['scope_size_exceeded']}")

print("\n5. TRACK_FUNC (node callback):")
print(f"   nodes evaluated: {eval_results['track_func_count']}")
print(f"   node types: {eval_results['track_func_types']}")

print("\n6. MODULE_DICT (custom modules):")
print(f"   math_utils: double(5) + triple(3) = {eval_results['module_math']}")
print(f"   string_utils: shout('hello') = {eval_results['module_string']}")

print("\n7. COMBINED (all parameters):")
print(f"   halve(base=100) = {eval_results['combined']}")
print(f"   nodes used: {eval_results['combined_nodes_used']}")

## Testing and Coverage

The dyslang module includes built-in tools for testing and code coverage analysis. By prefixing function names with `test_`, you can enable coverage mode, which provides detailed information about which parts of your code are being executed.

### Code Coverage Analysis

Let's create a script with a test function to demonstrate code coverage analysis:

In [ ]:
# Get Charlie's address
[CHARLIE_ADDRESS] = ! dysond keys show -a charlie

# Create a script with test coverage
coverage_script = '''
def a_or_b(a, b):
    if a:
        return a
    if b:
        return b
    return None

def coverage_a_or_b():
    # Test with different inputs
    a_or_b(1, 0)  # Should return a
    a_or_b(1, 1)  # Should still return a (first condition)
    # Note: We're not testing the b condition or the fallback
'''

# Save and execute the script
with open('/tmp/coverage_test.py', 'w') as f:
    f.write(coverage_script)

out = ! dysond q script run --script-address {CHARLIE_ADDRESS} --executor-address {CHARLIE_ADDRESS} --function-name coverage_a_or_b --extra-code-path /tmp/coverage_test.py -o json
out = '\n'.join(out)
print(out)
json_out = json.loads(out)
print(json.loads(json_out['result']))
# Extract and interpret the coverage data
coverage_data = json.loads(json_out['result'])['result']


In [ ]:
# Display a simplified analysis of the coverage data
print("Coverage Analysis Results:")
for item in coverage_data:
    node_info = item[0]
    count, memory_usage = item[1]
    line = node_info[0]
    node_type = node_info[4]
    
    # Simplify the coverage output for key lines
    if node_type == "FunctionDef" and line == 3:
        print(f"- FunctionDef (a_or_b): executed {count} time{'s' if count != 1 else ''} and used {memory_usage} bytes of memory")
    elif node_type == "If" and line == 4:
        print(f"- If (line {line}): {f'executed {count} times' if count > 0 else 'never executed'} and used {memory_usage} bytes of memory")
    elif node_type == "If" and line == 6:
        print(f"- If (line {line}): {f'executed {count} times' if count > 0 else 'never executed'} and used {memory_usage} bytes of memory")
    elif node_type == "Return" and line == 5:
        print(f"- Return (line {line}): {f'executed {count} times' if count > 0 else 'never executed'} and used {memory_usage} bytes of memory")
    elif node_type == "Return" and line == 7:
        print(f"- Return (line {line}): {f'executed {count} times' if count > 0 else 'never executed'} and used {memory_usage} bytes of memory")
    elif node_type == "Return" and line == 8:
        print(f"- Return (fallback): {f'executed {count} times' if count > 0 else 'never executed'} and used {memory_usage} bytes of memory")
assert len(coverage_data) > 0, "Coverage data should be greater than 0"

From the coverage analysis, we can see that:

1. The `a_or_b` function was defined (executed once)
2. The first `if` condition (line 4) was evaluated twice and passed both times
3. The first `return` statement (line 5) was executed twice
4. The second `if` condition (line 6) was never evaluated because the first condition always passed
5. The second `return` statement (line 7) was never executed
6. The fallback `return None` (line 8) was never executed

This coverage analysis helps us identify test gaps in our code. In this case, we need to add tests for when the first condition fails to ensure we're testing all code paths.

## Available Modules and Functions

These are the available modules and functions that can be used in dyslang scripts.

In [ ]:
# Render available modules and functions by executing dysond with --extra-code
import json
import tempfile
from IPython.display import HTML, display


extra_code = (
    "from dys import list_modules, list_functions\n"
    "def list_api():\n"
    "    return {\n"
    "        'list_modules': list_modules(),\n"
    "        'list_functions': list_functions(),\n"
    "    }\n"
)

with tempfile.NamedTemporaryFile("w+", suffix=".py", delete=True) as f:
    f.write(extra_code)
    f.flush()
    EXTRA_PATH = f.name

    out = ! dysond query script run --script-address {ALICE_ADDRESS} --executor-address {ALICE_ADDRESS} --function-name list_api --extra-code-path {EXTRA_PATH} -o json
    outer = json.loads("".join(out))
    inner = json.loads(outer["result"])["result"]

print(json.dumps(inner, indent=2))

## Available Syntax

Here is a table of all the python syntax that is supported by Dyslang.

In [ ]:
# Render a single HTML table of AST node compatibility by evaluating each snippet via dys_eval
import json
import subprocess
import tempfile
from IPython.display import HTML, display

# Minimal AST examples (mirrors examples/ast_explorer.py)
ast_examples = {
    # Basic literals and constants
    "Constant": "42",
    "FormattedValue": "f'The answer is {40 + 2}'",
    "JoinedStr": "f'Hello {\"world\"}'",
    # Collections
    "List": "[1, 2, 3]",
    "Tuple": "(1, 2, 3)",
    "Set": "{1, 2, 3}",
    "Dict": "{'a': 1, 'b': 2}",
    # Variables
    "Name_Load": "x = 1; x",
    "Name_Store": "x = 42",
    "Name_Del": "y = 10; del y",
    "Starred": "a, *b = [1, 2, 3, 4]; b",
    # Expressions
    "UnaryOp_Not": "not True",
    "UnaryOp_Invert": "~42",
    "UnaryOp_UAdd": "+42",
    "UnaryOp_USub": "-42",
    # Binary operations
    "BinOp_Add": "1 + 2",
    "BinOp_Sub": "1 - 2",
    "BinOp_Mult": "2 * 3",
    "BinOp_Div": "6 / 3",
    "BinOp_FloorDiv": "7 // 3",
    "BinOp_Mod": "7 % 3",
    "BinOp_Pow": "2 ** 3",
    "BinOp_LShift": "1 << 2",
    "BinOp_RShift": "8 >> 2",
    "BinOp_BitOr": "1 | 2",
    "BinOp_BitXor": "5 ^ 3",
    "BinOp_BitAnd": "5 & 3",
    "BinOp_MatMult": "# Not in basic Python: a @ b",
    # Boolean operations
    "BoolOp_And": "True and False",
    "BoolOp_Or": "True or False",
    # Comparisons
    "Compare_Eq": "1 == 1",
    "Compare_NotEq": "1 != 2",
    "Compare_Lt": "1 < 2",
    "Compare_LtE": "1 <= 2",
    "Compare_Gt": "2 > 1",
    "Compare_GtE": "2 >= 1",
    "Compare_Is": "1 is 1",
    "Compare_IsNot": "1 is not 2",
    "Compare_In": "1 in [1, 2, 3]",
    "Compare_NotIn": "0 not in [1, 2, 3]",
    # Function and method calls
    "Call": "len([1, 2, 3])",
    "Call_Kwargs": "dict(a=1, b=2)",
    "Call_Starred": "sum([1, 2, 3])",
    "Call_KwStarred": "dict(**{'a': 1, 'b': 2})",
    # Conditional expressions
    "IfExp": "1 if True else 2",
    # Attribute access
    "Attribute": "'hello'.upper()",
    # Subscripting
    "Subscript": "[1, 2, 3][0]",
    "Slice": "[1, 2, 3, 4][1:3]",
    # Comprehensions
    "ListComp": "[x for x in range(5)]",
    "SetComp": "{x for x in range(5)}",
    "DictComp": "{x: x*x for x in range(5)}",
    "GeneratorExp": "(x for x in range(5))",
    # Assignments
    "Assign": "x = 42",
    "AnnAssign": "x: int = 42",
    "AugAssign": "x = 1; x += 1",
    "NamedExpr": "(x := 42)",
    # Control flow
    "If": "if True: pass",
    "For": "for i in range(5): pass",
    "While": "while False: pass",
    "Break": "for i in range(5):\n    if i > 2: break",
    "Continue": "for i in range(5):\n    if i < 2: continue",
    # Exception handling
    "Try": "try:\n    1/0\nexcept ZeroDivisionError:\n    pass",
    "Raise": "try:\n    raise ValueError('example error')\nexcept ValueError:\n    pass",
    "Assert": "assert True, 'message'",
    # Function and class definitions
    "FunctionDef": "def func(x): return x*2",
    "Lambda": "lambda x: x*2",
    "Return": "def func(): return 42",
    "ClassDef": "class MyClass:\n    pass",
    # Import statements
    "Import": "try: import json\nexcept ImportError: pass",
    "ImportFrom": "try: from json import loads\nexcept ImportError: pass",
    # With statements
    "With": "with open('file.txt', 'w') as f: pass",
    # Async/await
    "AsyncFunctionDef": "async def func(): pass",
    "Await": "async def func():\n    await other_func()",
    "AsyncFor": "async def func():\n    async for i in aiter(): pass",
    "AsyncWith": "async def func():\n    async with acontext() as a: pass",
    # Yield expressions
    "Yield": "def gen(): yield 42",
    "YieldFrom": "def gen(): yield from [1, 2, 3]",
    # Others
    "Delete": "x = 1; del x",
    "Pass": "pass",
    "Global": "global x",
    "Nonlocal": "nonlocal x",
}

# Categories (mirrors grouping used by the DWapp)
categories = {
    "Literals and Constants": ["Constant", "FormattedValue", "JoinedStr"],
    "Collections": ["List", "Tuple", "Set", "Dict"],
    "Variables": ["Name_Load", "Name_Store", "Name_Del", "Starred"],
    "Expressions": [
        "UnaryOp_Not",
        "UnaryOp_Invert",
        "UnaryOp_UAdd",
        "UnaryOp_USub",
    ],
    "Binary Operations": [
        "BinOp_Add",
        "BinOp_Sub",
        "BinOp_Mult",
        "BinOp_Div",
        "BinOp_FloorDiv",
        "BinOp_Mod",
        "BinOp_Pow",
        "BinOp_LShift",
        "BinOp_RShift",
        "BinOp_BitOr",
        "BinOp_BitXor",
        "BinOp_BitAnd",
        "BinOp_MatMult",
    ],
    "Boolean Operations": ["BoolOp_And", "BoolOp_Or"],
    "Comparisons": [
        "Compare_Eq",
        "Compare_NotEq",
        "Compare_Lt",
        "Compare_LtE",
        "Compare_Gt",
        "Compare_GtE",
        "Compare_Is",
        "Compare_IsNot",
        "Compare_In",
        "Compare_NotIn",
    ],
    "Function and Method Calls": [
        "Call",
        "Call_Kwargs",
        "Call_Starred",
        "Call_KwStarred",
    ],
    "Conditional Expressions": ["IfExp"],
    "Attribute Access": ["Attribute"],
    "Subscripting": ["Subscript", "Slice"],
    "Comprehensions": ["ListComp", "SetComp", "DictComp", "GeneratorExp"],
    "Assignments": ["Assign", "AnnAssign", "AugAssign", "NamedExpr"],
    "Control Flow": ["If", "For", "While", "Break", "Continue"],
    "Exception Handling": ["Try", "Raise", "Assert"],
    "Function and Class Definitions": [
        "FunctionDef",
        "Lambda",
        "Return",
        "ClassDef",
    ],
    "Import Statements": ["Import", "ImportFrom"],
    "With Statements": ["With"],
    "Async/Await": ["AsyncFunctionDef", "Await", "AsyncFor", "AsyncWith"],
    "Yield Expressions": ["Yield", "YieldFrom"],
    "Others": ["Delete", "Pass", "Global", "Nonlocal"],
}

# Use previously defined ALICE_ADDRESS in this notebook
SCRIPT_ADDR = ALICE_ADDRESS


def evaluate_in_dys(code_str, script_addr):
    # Comments mark unsupported examples we intentionally skip
    if code_str.strip().startswith("#"):
        return {"status": "skipped", "message": ""}

    # Build a minimal extra-code file that evaluates the snippet
    extra_code = (
        "from dys import dys_eval\n"
        "def run():\n"
        f"    code = {json.dumps(code_str)}\n"
        "    try:\n"
        "        result = dys_eval(code)\n"
        "        return {\"status\": \"success\", \"result\": str(result)}\n"
        "    except NotImplementedError:\n"
        "        return {\"status\": \"error\", \"message\": \"Not Implemented\"}\n"
        "    except Exception as e:\n"
        "        raise Exception(\"Bug during dys_eval: \" + code + \" <error>\" + str(e) + \"</error>\")\n"
    )

    with tempfile.NamedTemporaryFile("w+", suffix=".py", delete=False) as f:
        f.write(extra_code)
        path = f.name

    proc = subprocess.run(
        [
            "dysond", "query", "script", "run",
            "--script-address", script_addr,
            "--executor-address", script_addr,
            "--function-name", "run",
            "--extra-code-path", path,
            "-o", "json",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    outer = json.loads(proc.stdout)
    inner = json.loads(outer["result"]).get("result", {})
    return inner if isinstance(inner, dict) else {"status": "error", "message": "unexpected result"}


def html_escape(s: str) -> str:
    return s.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

# Build single HTML table output
html_lines = []
html_lines.append("<table>\n")
html_lines.append("<thead><tr><th>AST Node</th><th>Demo</th><th>Result</th></tr></thead>\n")
html_lines.append("<tbody>\n")
for category, nodes in categories.items():
    html_lines.append(f"<tr><td colspan=\"3\"><h3>{html_escape(category)}</h3></td></tr>\n")
    for node in nodes:
        if node not in ast_examples:
            continue
        code = ast_examples[node]
        res = evaluate_in_dys(code, SCRIPT_ADDR)
        status = res.get("status", "unknown")
        if status == "success":
            result_text = f"SUCCESS: {res.get('result')}"
        elif status == "skipped":
            result_text = "SKIPPED"
        elif status == "error":
            result_text = f"ERROR: {res.get('message', '')}"
        else:
            result_text = "UNKNOWN"
        safe_code_html = html_escape(code)
        result_html = html_escape(result_text)
        html_lines.append(f"<tr><td>{node}</td><td><pre><code>{safe_code_html}</code></pre></td><td><pre><code>{result_html}</code></pre></td></tr>\n")
html_lines.append("</tbody></table>")

display(HTML("".join(html_lines)))


## Security Constraints

The Dyson Protocol implements several security constraints to ensure safe and reliable execution of scripts:

### Gas Limits

All script executions are bound by gas limits to prevent infinite loops and excessive computation. As we saw earlier, you can query the gas limit for any execution using `get_gas_limit()`.

### Memory Restrictions

The dyslang module enforces limits on string lengths, stack depth, and scope sizes to prevent resource exhaustion. You can monitor memory usage with `get_cumulative_size()`.

### Sandboxed Environment

Scripts run in a carefully controlled environment where only whitelisted functions and modules are available. This prevents access to potentially dangerous system functions.

### Node Calls Tracking

As we've seen, the execution environment tracks AST node evaluations and terminates if limits are exceeded, preventing resource-exhaustion attacks.